## 1. Criação do Catálogo (Unity Catalog)
O primeiro passo é criar um contêiner isolado para nossos dados.
* **Decisão:** Utilizamos o comando `CREATE CATALOG IF NOT EXISTS`. Isso garante a idempotência (o código não falha se o catálogo já existir).
* **Motivo:** Usar um catálogo dedicado (`medalhao_credit`) evita conflitos com o catálogo `default` do Databricks e organiza melhor a governança dos dados.

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao_credit;
-- usamos o nome medalhao_credit para evitar conflito com o catalogo default

## 2. Definição da Arquitetura Medalhão (Schemas)
Aqui definimos a estrutura lógica do banco de dados seguindo as melhores práticas da arquitetura Lakehouse.
* **Bronze:** Camada para dados brutos ("raw"), ingeridos exatamente como vieram da fonte, apenas convertidos para Delta.
* **Silver:** Camada para dados limpos, tratados e com tipos de dados corrigidos.
* **Contexto:** O comando `USE CATALOG` garante que todas as operações subsequentes ocorram dentro do nosso catálogo `medalhao_credit`.

In [0]:
%sql

USE CATALOG medalhao_credit;

CREATE SCHEMA IF NOT EXISTS bronze_credit;
CREATE SCHEMA IF NOT EXISTS silver_credit;

## 3. Configuração do Volume de Dados
O Unity Catalog utiliza **Volumes** para gerenciar arquivos não tabulares (como CSVs, JSONs, imagens).
* **Objetivo:** Criar um local (`data`) para armazenar os arquivos CSV brutos que faremos a ingestão.
* **Vantagem:** Volumes oferecem um caminho de acesso unificado e governado, substituindo os antigos "mount points" do DBFS.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS data;%sql
CREATE VOLUME IF NOT EXISTS data;

## 4. Configuração do Ambiente e Bibliotecas
Importação das bibliotecas essenciais do PySpark e Python padrão.

In [0]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

import requests
import pandas as pd
from datetime import datetime
import time

## 5. Definição de Variáveis Globais
Centralizamos os nomes de catálogos, bancos de dados e caminhos.
* **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.
* **Caminho do Volume:** Aponta para o local físico onde os CSVs estão armazenados dentro do Unity Catalog.

In [0]:
catalogo = "medalhao_credit"
bronze_db_name = "bronze_credit"

volume_path = "/Volumes/workspace/default/data"

## 6. Função de Ingestão (Bronze)
Esta função é o núcleo do nosso processo de ingestão para a camada Bronze.

**Decisões Técnicas:**
1.  **Parâmetro `tem_header`:** Flexibiliza a função para ler tanto arquivos que possuem cabeçalho quanto os que não possuem.
2.  **`inferSchema=True`:** Na camada Bronze, aceitamos a inferência automática para agilizar a ingestão. A tipagem forte será forçada na camada Silver.
3.  **Coluna `data_ingestao`:** Adicionamos metadados de auditoria para saber quando o dado entrou no Lakehouse.
4.  **Formato Delta:** Salvamos como tabela Delta para garantir transações ACID e performance.
5.  **`mergeSchema`:** Habilitado para permitir que a tabela evolua caso novos arquivos tragam colunas novas no futuro.

In [0]:
import pyspark.sql.functions as F

# 1. Adicionando o parâmetro 'tem_header' (padrao False para não quebrar os sem header)
def ingest_csv(nome_arquivo, nome_tabela, tem_header=False):
    
    try:
        landing_path = f'/Volumes/medalhao_credit/default/data/{nome_arquivo}'

        # 2. Passe a variável 'tem_header' para a opção do Spark
        # O inferSchema continua True para detectar tipos (int, date, etc)
        df = spark.read.csv(landing_path, header=tem_header, inferSchema=True)

        # Validação: arquivo vazio
        if df.count() == 0:
            # Nota: count() é uma ação custosa, se os arquivos forem gigantes, cuidado aqui.
            raise ValueError(f'O arquivo {nome_arquivo} está vazio ou não pôde ser lido.')

        # Adiciona timestamp de ingestão
        df_with_metadata = df.withColumn('data_ingestao', F.current_timestamp())

        # Escrita no formato Delta
        df_with_metadata.write \
            .format('delta') \
            .mode('overwrite') \
            .option("mergeSchema", "true") \
            .saveAsTable(f'{catalogo}.{bronze_db_name}.{nome_tabela}')

        print(f'Tabela bronze.{nome_tabela} criada com sucesso! (Header: {tem_header})\n')

    except Exception as e:
        print(f'Erro ao processar {nome_tabela}: {str(e)}')

## 7. Execução em Lote (Arquivos sem Cabeçalho)
Processamento dos arquivos que seguem o padrão "sem cabeçalho".
* **Automação:** Utilizamos listas e um loop `for` para processar múltiplos arquivos de uma vez, seguindo o princípio *DRY (Don't Repeat Yourself)*.
* **Padrão:** Todos estes arquivos (`base_atendentes`, `clientes`, etc.) serão lidos com `header=False` (padrão da função definida acima).

In [0]:
# criando tabelas no schema bronze

nomes_arquivos = [
    'base_atendentes.csv',
    'base_motivos.csv',
    'canais.csv',
    'chamados.csv',
    'clientes.csv',
    'custos.csv',
    'pesquisa_satisfacao.csv'
]

nomes_tabelas = [
    'base_atendentes',
    'base_motivos',
    'canais',
    'chamados',
    'clientes',
    'custos',
    'pesquisa_satisfacao'
]


for i in range(0, len(nomes_arquivos)):
    ingest_csv(nomes_arquivos[i], nomes_tabelas[i])

## 8. Ingestão Específica (Arquivo com Cabeçalho / Header)
Tratamento de exceções ou arquivos com formatos diferentes.
* **Diferença:** O arquivo `Chamados_Hora.CSV` possui cabeçalho, diferentemente dos anteriores.
* **Ação:** Chamamos a função explicitamente passando `True` para o parâmetro de header.

In [0]:
ingest_csv('Chamados_Hora.CSV', 'chamados_hora', True)

## 9. Validação dos Dados
Leitura da tabela recém-criada para garantir que a ingestão funcionou.

In [0]:
df_Chamados_Hora = spark.table(f'medalhao_credit.{bronze_db_name}.chamados_hora')
df_Chamados_Hora.limit(5).display()